# Cross-Modal Fusion

**Novel experiment** — no direct text analogue in the original Patchscopes paper.

**Research question:** At which LLM backbone layer do text tokens that *refer* to visual
entities absorb visual information through attention?

For example: given an image of an apple and the prompt "There is an apple on the table.
Describe it." — at which layer does the word "it" transition from being a generic pronoun
to encoding specific visual information about the apple?

**Method:**
1. Source: image + text prompt; extract hidden state of a referring text token (e.g. "it")
   using `modality='text'`
2. Target: text-only description prompt (e.g. `'The object being referred to is:'`)
3. Track how generated description changes from generic → visually specific across layers
4. The layer where generation first matches visual content = **fusion point**

**Expected finding:** Fusion happens in mid LLM backbone layers (~10–15 for a 32-layer model)
for simple objects, later for complex relational descriptions.

**Dataset needed:** Images with referring-expression + expected visual description pairs  
- OK-VQA: https://okvqa.allenai.org/  
- Or custom annotations (image, referring_text, target_description, text_token_index)


In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from general_utils import ModelAndTokenizer
from patchscopes_utils import (
    set_hs_patch_hooks_llava_batch,
    inspect_vlm,
    evaluate_visual_attribute_extraction_batch,
)

## 1. Load Model

In [ ]:
MODEL_NAME = "llava-hf/llava-1.5-7b-hf"

import torch
mt = ModelAndTokenizer(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device="cuda",
)
mt.set_hs_patch_hooks = set_hs_patch_hooks_llava_batch

print(mt)
print(f"is_vlm={mt.is_vlm}, num_layers={mt.num_layers}, num_visual_tokens={mt.num_visual_tokens}")

## 2. Load Dataset

Each row:
- `image_path`: path to image
- `source_prompt`: text prompt containing the referring expression
  e.g. `"USER: There is an apple on the table. Describe it. ASSISTANT:"`
- `text_token_index`: index (within the text portion of the sequence) of the referring token
  Note: position 0 in 'text' modality = first text token after the 576 visual tokens
- `target_prompt`: verbalization prompt  
  e.g. `"The object being referred to is:"`
- `ground_truth`: expected visual description (e.g. `"red apple"`)
- `object_category`: type of referring expression (simple, relational, etc.)

In [ ]:
# TODO: set path to your referring expression dataset
DATASET_PATH = "./preprocessed_data/cross_modal_fusion.tsv"

dataset_df = pd.read_csv(DATASET_PATH, sep="\t")
print(f"Loaded {len(dataset_df)} samples")
dataset_df.head()

## 3. Build Experiment DataFrame

For cross-modal fusion, we use `modality='text'` on the source side to extract
the hidden state of the referring text token (which has attended to visual tokens).

We sweep over `layer_source` only (using `layer_source == layer_target`).

In [ ]:
rows = []
for _, row in dataset_df.iterrows():
    for layer in range(mt.num_layers):
        rows.append({
            "image_path": row["image_path"],
            "prompt_source": row["source_prompt"],
            "prompt_target": row["target_prompt"],
            # text_token_index is relative to the text portion of the sequence
            "position_source": int(row["text_token_index"]),
            "position_target": -1,
            "layer_source": layer,
            "layer_target": layer,
            "modality": "text",  # extract from TEXT token that refers to visual entity
            "ground_truth": row["ground_truth"],
            "object_category": row.get("object_category", "unknown"),
        })

exp_df = pd.DataFrame(rows)
print(f"Experiment rows: {len(exp_df)}")

## 4. Run Evaluation

In [ ]:
results = evaluate_visual_attribute_extraction_batch(
    mt,
    exp_df,
    batch_size=32,
    max_gen_len=15,
    transform=None,
)

exp_df["generation"] = results["generations"]
exp_df["is_correct"] = results["is_correct"]

exp_df.to_csv("./results_cross_modal_fusion.csv", index=False)
print("Saved results.")

## 5. Visualize

### 5a. Accuracy vs. layer (fusion curve)

The layer at which accuracy first rises steeply is the **fusion point**.

In [ ]:
mean_by_layer = exp_df.groupby(["layer_source", "object_category"])["is_correct"].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
for cat, grp in mean_by_layer.groupby("object_category"):
    ax.plot(grp["layer_source"], grp["is_correct"], marker="o", label=cat)

ax.set_xlabel("LLM Backbone Layer")
ax.set_ylabel("Accuracy (ground truth in generation)")
ax.set_title("Cross-Modal Fusion: When Do Text Tokens Absorb Visual Information?")
ax.legend(title="Object Category")
plt.tight_layout()
plt.savefig("./cross_modal_fusion_curve.png", dpi=150)
plt.show()

### 5b. Example: generation at each layer for one image

In [ ]:
sample_row = dataset_df.iloc[0]
sample_image = Image.open(sample_row["image_path"]).convert("RGB")

print(f"Source: {sample_row['source_prompt']}")
print(f"Target: {sample_row['target_prompt']}")
print(f"Ground truth: {sample_row['ground_truth']}")
print()

for layer in range(0, mt.num_layers, 3):
    gen = inspect_vlm(
        mt,
        image=sample_image,
        prompt_source=sample_row["source_prompt"],
        prompt_target=sample_row["target_prompt"],
        layer_source=layer,
        layer_target=layer,
        position_source=int(sample_row["text_token_index"]),
        position_target=-1,
        modality="text",
        generation_mode=True,
        max_gen_len=15,
    )
    correct = sample_row["ground_truth"].lower() in gen.lower()
    marker = "✓" if correct else " "
    print(f"Layer {layer:2d} [{marker}]: {gen}")

### 5c. Fusion point distribution

For each sample, the fusion point is the first layer where the generation is correct.

In [ ]:
def find_fusion_point(group):
    """First layer where is_correct becomes True."""
    correct_layers = group[group["is_correct"] == True]["layer_source"]
    return correct_layers.min() if len(correct_layers) > 0 else np.nan

# Assign a sample ID based on image_path + position_source for grouping
exp_df["sample_id"] = exp_df["image_path"] + "_" + exp_df["position_source"].astype(str)
fusion_points = exp_df.groupby(["sample_id", "object_category"]).apply(find_fusion_point).reset_index()
fusion_points.columns = ["sample_id", "object_category", "fusion_layer"]
fusion_points = fusion_points.dropna()

fig, ax = plt.subplots(figsize=(8, 4))
for cat, grp in fusion_points.groupby("object_category"):
    ax.hist(grp["fusion_layer"], bins=mt.num_layers, alpha=0.6, label=cat)
ax.set_xlabel("Fusion Layer")
ax.set_ylabel("Count")
ax.set_title("Distribution of Cross-Modal Fusion Points")
ax.legend(title="Object Category")
plt.tight_layout()
plt.savefig("./cross_modal_fusion_points.png", dpi=150)
plt.show()

print("Mean fusion layer by category:")
print(fusion_points.groupby("object_category")["fusion_layer"].mean())